In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple MPS (M1 Max)")
else:
    device = torch.device("cpu")
    print("Device: CPU")

Device: Apple MPS (M1 Max)


In [4]:
DATA_DIR      = "../data/EuroSAT_RGB"

BATCH_SIZE    = 32
NUM_CLASSES   = 10
SEED          = 42

STAGE1_EPOCHS = 5
STAGE1_LR     = 0.01

STAGE2_EPOCHS = 25
STAGE2_LR     = 0.0001

In [6]:
MEAN = [0.3444, 0.3803, 0.4078]
STD  = [0.2025, 0.1365, 0.1148]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

In [8]:
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
class_names = full_dataset.classes

total = len(full_dataset)
val_size = int(total * 0.2)
train_size = total - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=generator
)

val_dataset.dataset = datasets.ImageFolder(root=DATA_DIR, transform=val_transform)

print(f"Total images: {total}")
print(f"Training images: {train_size}")
print(f"Validation images: {val_size}")
print(f"Classes: {class_names}")

Total images: 27000
Training images: 21600
Validation images: 5400
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)